In [1]:
# KNN regressor from scratch — House Price Predictor
import numpy as np
import pandas as pd
from math import sqrt

# ------------------------
# Helper utilities
# ------------------------
def train_test_split_custom(X, y, test_size=0.2, shuffle=True, seed=42):
    np.random.seed(seed)
    m = X.shape[0]
    idx = np.arange(m)
    if shuffle:
        np.random.shuffle(idx)
    split = int(m * (1 - test_size))
    train_idx = idx[:split]
    test_idx = idx[split:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def normalize_train_test(X_train, X_test):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1.0
    X_train_n = (X_train - mu) / sigma
    X_test_n = (X_test - mu) / sigma
    return X_train_n, X_test_n, mu, sigma

def mse(y_true, y_pred): return np.mean((y_true - y_pred)**2)
def mae(y_true, y_pred): return np.mean(np.abs(y_true - y_pred))
def rmse(y_true, y_pred): return sqrt(mse(y_true, y_pred))
def r2_score_custom(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ss_res/ss_tot

# ------------------------
# Distance functions
# ------------------------
def euclidean(a, b):
    # a and b are 1D arrays of same length
    return np.sqrt(np.sum((a - b)**2))

# ------------------------
# KNN regressor implementation
# ------------------------
class KNNRegressorFromScratch:
    def __init__(self, k=5, weighted=False):
        """
        k: number of neighbors
        weighted: if True, weight neighbors by inverse distance (1/(d+eps))
        """
        self.k = int(k)
        self.weighted = bool(weighted)
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        """Store training data (KNN is a lazy learner)."""
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def _predict_single(self, x):
        """Predict for a single example x (1D array)."""
        # compute distances to all training points
        dists = np.sqrt(np.sum((self.X_train - x)**2, axis=1))
        # get indices of k smallest distances
        k = min(self.k, len(dists))
        idx = np.argsort(dists)[:k]
        neigh_y = self.y_train[idx]
        neigh_d = dists[idx]
        if self.weighted:
            # avoid division by zero
            eps = 1e-8
            weights = 1.0 / (neigh_d + eps)
            return np.sum(weights * neigh_y) / np.sum(weights)
        else:
            return np.mean(neigh_y)

    def predict(self, X):
        """Predict for multiple examples (2D array)."""
        X = np.array(X)
        preds = [self._predict_single(x) for x in X]
        return np.array(preds)

# ------------------------
# k selection using validation set
# ------------------------
def select_best_k(X_train, y_train, X_val, y_val, k_list=[1,3,5,7,9], weighted=False):
    best_k = None
    best_rmse = float('inf')
    results = []
    for k in k_list:
        model = KNNRegressorFromScratch(k=k, weighted=weighted)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        curr_rmse = rmse(y_val, y_pred)
        results.append((k, curr_rmse))
        if curr_rmse < best_rmse:
            best_rmse = curr_rmse
            best_k = k
    return best_k, best_rmse, results

# ------------------------
# Example usage with synthetic data (or replace with your CSV)
# ------------------------
if __name__ == "__main__":
    # --------------------
    # 1) Data — replace this block with your CSV if you have one
    # --------------------
    # Synthetic demo (same features as earlier): area, bedrooms, age, distance
    np.random.seed(42)
    n_samples = 500
    area = np.random.normal(1200, 350, n_samples)
    bedrooms = np.clip(np.round(np.random.normal(3, 0.8, n_samples)), 1, 6)
    age = np.clip(np.round(np.random.exponential(20, n_samples)), 0, 100)
    distance = np.abs(np.random.normal(8, 6, n_samples))
    true_w = np.array([120.0, 8000.0, -150.0, -300.0])
    bias_true = 50000.0
    noise = np.random.normal(0, 20000, n_samples)
    X_full = np.vstack([area, bedrooms, age, distance]).T
    y_full = X_full @ true_w + bias_true + noise
    df = pd.DataFrame(X_full, columns=['area','bedrooms','age','distance'])
    df['price'] = y_full

    # If you want to use your CSV, uncomment and adapt:
    # df = pd.read_csv("your_house_data.csv")
    # X_full = df[['area','bedrooms','age','distance']].values
    # y_full = df['price'].values

    # --------------------
    # 2) Train/Val/Test split (60/20/20)
    # --------------------
    X = df[['area','bedrooms','age','distance']].values
    y = df['price'].values
    X_trainval, X_test, y_trainval, y_test = train_test_split_custom(X, y, test_size=0.2, shuffle=True, seed=42)
    X_train, X_val, y_train, y_val = train_test_split_custom(X_trainval, y_trainval, test_size=0.25, shuffle=True, seed=24)

    # --------------------
    # 3) Normalize (important for distance methods)
    # --------------------
    X_train_n, X_val_n, mu, sigma = normalize_train_test(X_train, X_val)
    X_test_n = (X_test - mu) / sigma

    # --------------------
    # 4) Select best k (try unweighted and weighted if you want)
    # --------------------
    k_candidates = [1,3,5,7,9,11,15]
    best_k, best_rmse, trial_results = select_best_k(X_train_n, y_train, X_val_n, y_val, k_list=k_candidates, weighted=False)
    print("K selection (unweighted):", trial_results)
    print(f"Best k (unweighted): {best_k} with val RMSE={best_rmse:,.2f}")

    # Optionally try weighted
    best_k_w, best_rmse_w, trial_results_w = select_best_k(X_train_n, y_train, X_val_n, y_val, k_list=k_candidates, weighted=True)
    print("K selection (weighted):", trial_results_w)
    print(f"Best k (weighted): {best_k_w} with val RMSE={best_rmse_w:,.2f}")

    # --------------------
    # 5) Train final model on train+val (recommended) and evaluate on test
    # --------------------
    # Combine train + val for final model
    X_train_final = np.vstack([X_train, X_val])
    y_train_final = np.concatenate([y_train, y_val])
    # normalize using combined training stats
    X_train_final_n, X_test_final_n, mu_final, sigma_final = normalize_train_test(X_train_final, X_test)

    final_k = best_k  # choose weighted/unweighted based on earlier results
    final_model = KNNRegressorFromScratch(k=final_k, weighted=False)
    final_model.fit(X_train_final_n, y_train_final)
    y_pred_test = final_model.predict(X_test_final_n)

    # --------------------
    # 6) Metrics & quick summary
    # --------------------
    print("=== Test set performance (final KNN) ===")
    print(f"K = {final_k}")
    print(f"MSE:  {mse(y_test, y_pred_test):,.2f}")
    print(f"MAE:  {mae(y_test, y_pred_test):,.2f}")
    print(f"RMSE: {rmse(y_test, y_pred_test):,.2f}")
    print(f"R2:   {r2_score_custom(y_test, y_pred_test):.4f}")

    # Example: show first 10 predictions vs actual
    print("\\nFirst 10: Actual vs Predicted")
    for a, p in list(zip(y_test[:10], y_pred_test[:10])):
        print(f"Actual: {a:,.0f}  Pred: {p:,.0f}")


K selection (unweighted): [(1, 27517.67895019018), (3, 25126.23657649619), (5, 25619.23305064828), (7, 25594.60362333445), (9, 26091.644387230597), (11, 26824.36744939869), (15, 26888.150635892674)]
Best k (unweighted): 3 with val RMSE=25,126.24
K selection (weighted): [(1, 27517.678950190177), (3, 25130.542260497536), (5, 25184.50982042472), (7, 25249.716871217985), (9, 25594.47440598953), (11, 26134.45110238626), (15, 26187.10388212983)]
Best k (weighted): 3 with val RMSE=25,130.54
=== Test set performance (final KNN) ===
K = 3
MSE:  638,743,729.80
MAE:  19,325.07
RMSE: 25,273.38
R2:   0.6671
\nFirst 10: Actual vs Predicted
Actual: 234,585  Pred: 238,309
Actual: 192,485  Pred: 209,034
Actual: 180,364  Pred: 176,014
Actual: 272,777  Pred: 240,132
Actual: 292,258  Pred: 289,225
Actual: 173,632  Pred: 180,311
Actual: 236,088  Pred: 236,183
Actual: 262,222  Pred: 232,088
Actual: 251,338  Pred: 234,216
Actual: 245,655  Pred: 240,890
